In [ ]:
# run this notebook after 29_rev1_lig1_sibling_compare_R
# use this notebook to test for a rev1-replication timing relationship
# use this notebook to test for a lig1-methylation relationship
# we need mutations for lig1 and rev1 carriers and non-carriers from ukb (./ukb/LIG1_carriers_dnms.csv, 
# ./ukb/LIG1_noncarriers_dnms.csv, ./ukb/REV1_carriers_dnms.csv, ./ukb/REV1_noncarriers_dnms.csv)

In [ ]:
library(data.table)
library(dplyr)
library(ggplot2)
library(tidyr)

In [ ]:
lof = fread("./gts/aou_gt_lof_indel_snps_ages_added.txt")
missense = fread("./gts/aou_gt_missense_ages_added.txt")
missense_rev1 = missense %>% filter(REV1 >= 1) %>% select(fid)
missense_lig1 = missense %>% filter(LIG1 >= 1) %>% select(fid)
lof_rev1 = lof %>% filter(REV1 >= 1) %>% select(fid)
lof_lig1 = lof %>% filter(LIG1 >= 1) %>% select(fid)
rev1_c = unique(rbind(missense_rev1, lof_rev1))
lig1_c = unique(rbind(missense_lig1, lof_lig1))

In [ ]:
sib_difs = fread("./sibs_snps_QC_GIAB_distance_AF_outliers_COSMIC.txt")
trio_difs = fread("./trios_snps_after_standard_QC_GIAB_COSMIC.txt")

In [ ]:
lig1_carriers = sib_difs %>% filter(fid %in% lig1_c$fid & 
                                    substr(Type, 3,7) == "C>T]G") %>% select(chr, pos)
lig1_carriers = lig1_carriers %>% rbind(trio_difs %>% filter(fid %in% lig1_c$fid & 
                                    substr(Type, 3,7) == "C>T]G") %>% select(chr, pos))
lig1_carriers$type = "carrier"
lig1_noncarriers = sib_difs %>% filter((!fid %in% lig1_c$fid) & 
                                    substr(Type, 3,7) == "C>T]G") %>% select(chr, pos)

lig1_noncarriers = lig1_noncarriers %>% rbind(trio_difs %>% filter((!fid %in% lig1_c$fid) & 
                                    substr(Type, 3,7) == "C>T]G") %>% select(chr, pos))
lig1_noncarriers$type = "noncarrier"

lig1_ukb_carrier = fread("./ukb/LIG1_carriers_dnms.csv", header=TRUE) %>% mutate(
chr = chrom, pos = pos, type = "carrier") %>% filter(is_cpg_tpg == 1) %>% select(chr, pos, type)
lig1_ukb_noncarrier = fread("./ukb/LIG1_noncarriers_dnms.csv", header = TRUE) %>% mutate(
chr = chrom, pos = pos, type = "noncarrier") %>% filter(is_cpg_tpg == 1) %>% select(chr, pos, type)
lig1 = rbind(lig1_carriers, lig1_noncarriers) %>% rbind(lig1_ukb_carrier) %>% rbind(lig1_ukb_noncarrier)
lig1$chr = paste0("chr", lig1$chr)
fwrite(lig1, "./lig1_cpg_dnms.txt", sep = "\t", 
      row.names = FALSE, col.names = TRUE, quote = FALSE)

In [ ]:
# run this in python kernel 

# import gzip
# import pandas as pd

# # ── config ────────────────────────────────────────────────────────────────────
# WGBS_FILE  = "./cpg/ENCFF053FUK.bed.gz"
# CPG_FILE   = "./lig1_cpg_dnms.txt"       # your cpg dataframe
# OUTFILE    = "./cpg/lig1_cpg_dnms_with_meth.tsv"
# METH_COL   = 10
# AUTOSOMES  = {f"chr{i}" for i in range(1, 23)}
# # ─────────────────────────────────────────────────────────────────────────────

# # ── load mutations ────────────────────────────────────────────────────────────
# cpg = pd.read_csv(CPG_FILE, sep="\t")
# cpg = cpg[cpg["chr"].isin(AUTOSOMES)]

# # ── stream WGBS file and extract methylation at mutation positions ─────────────
# # index mutations by (chr, pos) for fast lookup
# mut_index = set(zip(cpg["chr"], cpg["pos"] - 1))

# meth_lookup = {}  # (chr, pos) -> methylation %

# open_fn = gzip.open if WGBS_FILE.endswith(".gz") else open
# with open_fn(WGBS_FILE, "rt") as fh:
#     for line in fh:
#         if line.startswith(("#", "track", "browser")):
#             continue
#         cols = line.rstrip().split("\t")
#         if len(cols) <= METH_COL:
#             continue
#         try:
#             chrom = cols[0]
#             pos   = int(cols[1])
#             meth  = float(cols[METH_COL])
#         except ValueError:
#             continue
#         if (chrom, pos) in mut_index:
#             meth_lookup[(chrom, pos)] = meth

# print(f"Matched {len(meth_lookup):,} of {len(cpg):,} mutations to WGBS data")

# # ── join back to mutations ────────────────────────────────────────────────────
# cpg["methylation"] = cpg.apply(lambda r: meth_lookup.get((r["chr"], r["pos"] - 1), None), axis=1)
# cpg.to_csv(OUTFILE, sep="\t", index=False)
# print(f"Done! Written to {OUTFILE}")

In [ ]:
lig1_meth = fread("./cpg/lig1_cpg_dnms_with_meth.tsv")
ks.test(lig1_meth$methylation[lig1_meth$type == "carrier"], 
       lig1_meth$methylation[lig1_meth$type == "noncarrier"])
p_ecdf = ggplot(lig1_meth, aes(x = methylation, color = type)) +
  stat_ecdf(geom = "step", linewidth = 1.2) +
  scale_color_manual(values = c("carrier" = "darkgreen", "noncarrier" = "darkred")) +
  labs(
    x = "Methylation",
    y = "Cumulative proportion of mutations",
    color = "LIG1"
  ) +
  theme_bw()
p_ecdf

In [ ]:
rev1_carriers = sib_difs %>% filter(fid %in% rev1_c$fid ) %>% select(chr, pos)
rev1_carriers = rev1_carriers %>% rbind(trio_difs %>% filter(fid %in% rev1_c$fid ) %>% select(chr, pos))
rev1_carriers$type = "carrier"

rev1_noncarriers = sib_difs %>% filter((!fid %in% rev1_c$fid) ) %>% select(chr, pos)

rev1_noncarriers = rev1_noncarriers %>% rbind(trio_difs %>% filter((!fid %in% rev1_c$fid) ) %>% select(chr, pos))
rev1_noncarriers$type = "noncarrier"

# read ukb 
rev1_ukb_carrier = fread("./ukb/REV1_carriers_dnms.csv", header=TRUE) %>% mutate(
chr = chrom, pos = pos, type = "carrier") %>% select(chr, pos, type)
rev1_ukb_noncarrier = fread("./ukb/REV1_noncarriers_dnms.csv", header = TRUE) %>% mutate(
chr = chrom, pos = pos, type = "noncarrier") %>% select(chr, pos, type)
rev1 = rbind(rev1_carriers, rev1_noncarriers) %>% rbind(rev1_ukb_carrier) %>% rbind(rev1_ukb_noncarrier)
rev1$chr = paste0("chr", rev1$chr)

In [ ]:
rt = fread("./rt_windows.bed")
names(rt)[2:3] = c("start", "end")
rt$chr = paste0("chr", rt$chr)
opp = fread("./rt_windows_ca_opportunity_giab_masked.tsv")
rt = merge(rt, opp, by = c("chr", "start", "end"))

In [ ]:
rev1_carrier = rev1 %>% filter(type == "carrier")
rev1_noncarrier = rev1 %>% filter(type == "noncarrier")
carrier_bed = data.frame(chrom = rev1_carrier$chr, start = rev1_carrier$pos, end = rev1_carrier$pos + 1)
noncarrier_bed = data.frame(chrom = rev1_noncarrier$chr, start = rev1_noncarrier$pos, end = rev1_noncarrier$pos + 1)
rt_bed = data.frame(chrom = rt$chr, start = rt$start, end = rt$end, timing = rt$mean_rt, opp = rt$unmasked_bp, bin = rt$rt_rank)
carrier_merged = bedtoolsr::bt.intersect(rt_bed, carrier_bed, c=TRUE)
noncarrier_merged = bedtoolsr::bt.intersect(rt_bed, noncarrier_bed, c=TRUE)

In [ ]:
timing_combined = bind_rows(
  carrier_merged    %>% mutate(group = "carrier"),
  noncarrier_merged %>% mutate(group = "noncarrier")
) %>%
  filter(!is.na(timing))

In [ ]:
ks.test(timing_combined$mean_rt[timing_combined$type == "carrier"], 
       timing_combined$mean_rt[timing_combined$type == "noncarrier"])

In [ ]:
p_ecdf = ggplot(timing_combined, aes(x = timing, color = group)) +
  stat_ecdf(geom = "step", linewidth = 1.2) +
  scale_color_manual(values = c("carrier" = "darkgreen", "noncarrier" = "darkred")) +
  labs(
    x = "Replication timing",
    y = "Cumulative proportion of mutations",
    color = "REV1"
  ) + lims( x = c(-3, 3)) + scale_x_continuous(breaks = seq(-3, 3, by = 1), limits = c(-3, 3)) + 
  theme_bw()

p_ecdf